# 🦺 SafetyCite on Colab — OSHA Compliance Copilot (fully live)

Runs the **entire** project on Colab's free GPU: pulls real OSHA regulations, trains per-domain LoRA adapters (SFT + GRPO with a verifiable citation reward), serves the real model behind a FastAPI + React UI, and exposes it with a **public cloudflared URL** you can open and share.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (an L4/A100 high-RAM runtime is nicer for GRPO on the 4B model).

**Model:** defaults to **`Qwen/Qwen3-4B-Instruct-2507`** — far more accurate than the 0.6B local default, and fits a 16 GB T4 in bf16. On a T4, SFT is quick; GRPO on a 4B model is slower (lower `RL_STEPS` or set `RUN_RL=False` if you're impatient, or use a bigger runtime).

> The Colab session is ephemeral (the URL dies when the session ends). You can also flip the model engine to remote **MinT** GPUs — see the last cell.

In [ ]:
!nvidia-smi -L || echo 'No GPU — set Runtime → Change runtime type → T4 GPU'

## 1. Configure

In [ ]:
import os

REPO_URL = "https://github.com/olsisadiku/safetycite.git"
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"   # bigger = more accurate; fits a T4 in bf16
TRAIN_DOMAINS = ["construction", "general_industry", "recordkeeping"]
RUN_RL = True     # GRPO after SFT (optimises the verifiable citation reward)
RL_STEPS = 25     # GRPO steps per domain (4B GRPO is slow on a T4 — keep modest)

os.environ["SAFETYCITE_BASE_MODEL"] = BASE_MODEL   # every command below uses this model
print("model:", BASE_MODEL)

## 2. Clone + install

In [ ]:
if not os.path.exists('safetycite_repo'):
    !git clone {REPO_URL} safetycite_repo
%cd safetycite_repo
!pip install -q -e ".[local]"

## 3. Fetch the real OSHA corpus + build datasets

In [ ]:
!safetycite fetch
!safetycite build

## 4. Train the adapters (SFT → GRPO) on the GPU
GRPO samples a group of answers per question, scores each with the verifiable citation reward, and pushes the policy toward the better-cited ones.

In [ ]:
for d in TRAIN_DOMAINS:
    print(f'\n===== {d}: SFT ({BASE_MODEL}) =====')
    !safetycite sft {d}
    if RUN_RL:
        print(f'\n===== {d}: GRPO =====')
        !safetycite rl {d} --steps {RL_STEPS}

## 5. Evaluate base vs adapter (the honest delta)

In [ ]:
!safetycite eval all

## 6. Build the web UI (Bun + React + Vite + Tailwind)

In [ ]:
!curl -fsSL https://bun.sh/install | bash
os.environ['PATH'] = os.path.expanduser('~/.bun/bin') + ':' + os.environ['PATH']
!cd web && bun install && bun run build

## 7. Serve live + open the public URL 🚀

In [ ]:
import subprocess, time, re, urllib.request

# Start the API server (loads the real model + serves the built UI)
server = subprocess.Popen(['safetycite', 'serve', '--host', '0.0.0.0', '--port', '8000'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Starting server (loading model)…')
up = False
for _ in range(180):
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=2); up = True; break
    except Exception:
        time.sleep(2)
print('server up' if up else 'server failed — check logs')

# cloudflared quick tunnel -> public https URL
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n🦺  SafetyCite is LIVE at:', url)
print('   (keep this cell running; the link dies when the Colab session ends)')

## Optional: use remote MinT GPUs instead of Colab's
MinT is Tinker-compatible; the `tinker` SDK is a thin client and training runs on Macaron's servers.
```python
import os
os.environ['SAFETYCITE_BACKEND'] = 'mint'
os.environ['SAFETYCITE_BASE_MODEL'] = 'Qwen/Qwen3-4B-Instruct-2507'
os.environ['TINKER_API_KEY'] = 'sk-mint-...'   # from macaron.im/mindlab/mint
os.environ['TINKER_BASE_URL'] = 'https://mint.macaron.xin/'
!pip install -q tinker==0.6.3
```
Then re-run the train / eval / serve cells with `--backend mint`. On MinT you can also go bigger: `Qwen/Qwen3-30B-A3B-Instruct-2507`.